## AM demodulation and audio playback from a complex baseband IQ signal

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.fft as spfft
from scipy.io import wavfile
import scipy.signal as signal
import sounddevice as sd

In [2]:
# Interactive plotting.  Comment out this next line if inline plots are desired.
%matplotlib qt

In [3]:
# Function to plot the frequency domain spectrum of a complex signal.
#
# Inputs: 
#  y - complex time domain signal to be plotted
#  fs - sampling rate, Hz
#  ttext - title of plot
#  xlim - x axis plot limits: (min, max)
#  ylim - y axis plot limits: (min, max)
#
# Returns:
#  Plot of frequency domain representation of signal.
#
# Affects: None
#
# Exceptions: None
#
def spec_plot_complex(y, fs, ttext, xlim, ylim):
    delta_t = 1.0 / fs
    no_samps = len(y)
    yf = spfft.fft(y)
    xf = spfft.fftfreq(no_samps, delta_t)
    xf_shift = spfft.fftshift(xf)
    yf_shift = spfft.fftshift(yf)
    plt.figure()
    plt.plot(xf_shift, 10 * np.log10(np.abs(yf_shift)))
    plt.xlim(xlim)
    plt.ylim(ylim)
    plt.xlabel('Frequency, Hz')
    plt.ylabel('Spectral amplitude')
    plt.title(ttext)
    plt.grid()
    plt.show()

In [4]:
# Function to plot the frequency domain spectrum of a real signal.
#
# Inputs: 
#  y - complex time domain signal to be plotted
#  fs - sampling rate, Hz
#  ttext - title of plot
#  xlim - x axis plot limits: (min, max)
#  ylim - y axis plot limits: (min, max)
#
# Returns:
#  Plot of frequency domain representation of signal.
#
# Affects: None
#
# Exceptions: None
#
def spec_plot_real(y, fs, ttext, xlim, ylim):
    delta_t = 1.0 / fs
    no_samps = len(y)
    yf = spfft.rfft(y)
    xf = spfft.rfftfreq(no_samps, delta_t)
    plt.figure()
    plt.plot(xf, 10 * np.log10(np.abs(yf)))
    plt.xlim(xlim)
    plt.ylim(ylim)
    plt.xlabel('Frequency, Hz')
    plt.ylabel('Spectral amplitude')
    plt.title(ttext)
    plt.grid()
    plt.show()

In [5]:
# Sample AM modulated baseband recording of a WWV origin signal.
# In principle, any recording of an AM modulated signal can be substituted here, as long as 
#  the recording is formatted as two channels (stereo) where channel 1 = I data
#  and channel 2 = Q data.
# See "scipy.wavfile" function documentation for information on manipulating WAV files in python.
file_path = "HamSCI_AM_modulated_WWV_iq.wav"

In [6]:
# Read WAV file, convert to complex IQ format, determine signal sampling rate
fs, data_n = wavfile.read(file_path)
iq =  data_n[:,0] + 1j * data_n[:,1]

In [7]:
# plot IQ signal time series 
plt.figure()
plt.plot(np.real(iq),label='I')
plt.plot(np.imag(iq),label='Q')
plt.grid()
plt.legend()
plt.title('Recorded WWV IQ signal')
plt.xlabel("Sample Number")
plt.ylabel("Amplitude")
plt.show()

In [8]:
# plot frequency spectrum of complex IQ data, expressed logarithmically in dB
ylim = (-50, 50)
xlim = (-8000, 8000)
spec_plot_complex(iq, fs, 'Frequency Spectrum of Recorded WWV IQ Signal',xlim,ylim)

In [9]:
# Demodulate the AM information content in the complex IQ signal
demod_am = np.abs(iq) # For AM demodulation, simply taking the IQ magnitude performs the average

In [10]:
# Remove DC component from demodulated signal
demod_ave = np.average(demod_am)
demod_base = demod_am - demod_ave

In [11]:
# display time series of AM demodulated signal
plt.figure()
plt.plot(demod_base,label='Demodulated AM')
plt.grid()
plt.legend()
plt.title('AM Demodulated Signal')
plt.xlabel("Sample Number")
plt.ylabel("Amplitude")
plt.show()

In [12]:
# plot frequency spectrum of real AM demodulated data, expressed logarithmically in dB
ylim = (-50, 50)
xlim = (0, 8000)
spec_plot_real(demod_base, fs, 'Freq Spectrum AM Demodulated Signal', xlim, ylim)

In [13]:
# play AM demodulated signal through computer audio
sd.play(demod_base, fs)

In [14]:
# execute this cell to stop audio play
sd.stop()

In [15]:
# save the demodulated AM signal as an audio WAV file
wavfile.write("HamSCI_AM_demodulated_WWV.wav", fs, demod_am)